[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_transcriptomics.ipynb) [![Open In nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_transcriptomics.ipynb)

# Transcriptomics

Predict age from public RNA-seq datasets.

- [C. elegans: BiT Age](#worms)
- [Mammals: tAge and tAgeMortality](#mammals)

In [1]:
import pandas as pd
import pyaging as pya

<a id="worms"></a>

## Worms

Load the [GSE65765](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE65765) counts-per-million example from the [BiT Age paper](https://doi.org/10.1111/acel.13320).

In [2]:
example_path = pya.data.download_example_data("GSE65765", verbose=False)
df = pd.read_pickle(example_path)
df.iloc[:5, :5]

,WBGene00197333,WBGene00198386,WBGene00015153,WBGene00002061,WBGene00255704
SRR1793993,0.0,0.0,3.780174,169.240815,1.907427
SRR1793991,0.0,0.0,0.510354,412.628597,0.061861
SRR1793994,0.0,0.0,4.718708,274.733671,1.234644
SRR1793992,0.0,0.0,2.389905,351.612558,0.505892


### Predict age

BiT Age expects CPM values. Convert the matrix to AnnData, then run the clock.

In [3]:
adata = pya.pp.df_to_adata(df, verbose=False)
pya.pred.predict_age(adata, "BiTAge", verbose=False)
adata.obs.head()

,bitage
SRR1793993,182.353658
SRR1793991,27.337245
SRR1793994,241.629584
SRR1793992,32.178003


### Get citation

Predictions are in `.obs`; coverage and clock metadata are in `.uns`.

In [4]:
adata.uns["bitage_metadata"]

{'clock_name': 'bitage',
 'data_type': 'transcriptomics',
 'species': 'Caenorhabditis elegans',
 'year': 2021,
 'approved_by_author': '✅',
 'citation': 'Meyer, David H., and Björn Schumacher. "BiT age: A transcriptome-based aging clock near the theoretical limit of accuracy." Aging Cell 20 (2021): e13320.',
 'doi': 'https://doi.org/10.1111/acel.13320',
 'notes': 'Binarized whole-organism C. elegans RNA-seq clock that estimates temporally rescaled biological age; the released linear predictor sums coefficients for genes binarized on plus a 103.55-hour intercept.',
 'research_only': None,
 'tissue': ['whole organism'],
 'predicts': ['biological age'],
 'training_target': ['biological age'],
 'unit': ['hours'],
 'model_type': 'elastic net regression',
 'platform': ['RNA-seq'],
 'population': 'Caenorhabditis elegans',
 'journal': 'Aging Cell',
 'last_author': 'Björn Schumacher',
 'n_features': 576,
 'citations': 184,
 'citations_date': '2026-10-02',
 'version': '0.5.7',
 'preprocess': 'bin

<a id="mammals"></a>

## Mammals

For tAge and tAgeMortality, use the [authors' mouse example](https://gladyshev-lab.github.io/tAge/articles/tage-bulk.html) with 24 kidney and muscle samples from wild-type and Klotho-knockout mice. Load the original counts and metadata from a fixed source version. [Source details and licence](https://github.com/lucascamillomd/pyaging/blob/main/tutorials/data/tage_mouse_README.md).

In [5]:
source = "https://raw.githubusercontent.com/Gladyshev-Lab/tAge/0dba58fba356fecfbbb7c6f0cb27ced59ee6f99f/inst/extdata/"
counts = pd.read_csv(source + "Exprs_example.csv", index_col=0).T
metadata = pd.read_csv(source + "Metadata_example.csv", index_col=0)
assert counts.index.equals(metadata.index)
cohort = counts.join(metadata, validate="one_to_one")
cohort.groupby(["Tissue", "Genotype"]).size()

Tissue           Genotype 
Kidney           Klotho KO    6
                 WT           6
Skeletal muscle  Klotho KO    6
                 WT           6
dtype: int64

### Predict within each tissue

Use raw counts. pyaging maps genes and normalizes the cohort internally. Keep `mouse` as a feature and sample annotations in `.obs`. Score each tissue separately, with its wild-type samples as the reference.

In [6]:
clock_names = ["tage", "tagemortality"]
results = []
coverage = []

for tissue, frame in cohort.groupby("Tissue", sort=False):
    frame = frame.copy()
    frame["mouse"] = 1
    tissue_adata = pya.pp.df_to_adata(
        frame, metadata_cols=list(metadata.columns), verbose=False
    )
    tissue_adata.obs["tage_reference_group"] = tissue_adata.obs["Genotype"].eq("WT")
    pya.pred.predict_age(tissue_adata, clock_names, verbose=False)
    results.append(tissue_adata.obs[["Tissue", "Genotype", *clock_names]])
    coverage.append({
        "tissue": tissue,
        **{name: tissue_adata.uns[f"{name}_percent_na"] for name in clock_names},
    })

results = pd.concat(results)
results.groupby(["Tissue", "Genotype"])[clock_names].mean()

tage  tagemortality
Tissue          Genotype                          
Kidney          Klotho KO  7.818586       0.752386
                WT         0.937060       0.108283
Skeletal muscle Klotho KO  8.947811       0.640506
                WT         0.401220       0.026337

`tage` returns relative age in mouse months; `tagemortality` returns log10 mortality hazard ratios. Results depend on the cohort and reference group. This small cohort demonstrates the workflow.

For rat, macaque or human data, use that species' gene identifiers and indicator. Rescale only `tage` using the species lifespan factors in its metadata. The tAge clocks are for non-commercial academic research.

### Inspect coverage and metadata

Check missing model features before interpreting predictions. Missing genes use training reference values.

In [7]:
pd.DataFrame(coverage).set_index("tissue").rename_axis(columns="missing features (%)")

missing features (%),tage,tagemortality
tissue,,
Kidney,4.443597,4.443597
Skeletal muscle,10.212644,10.212644


In [8]:
tissue_adata.uns["tage_metadata"]

{'clock_name': 'tage',
 'data_type': 'transcriptomics',
 'species': 'multiple species',
 'year': 2026,
 'approved_by_author': '⌛',
 'citation': 'Tyshkovskiy, Alexander, et al. "Universal transcriptomic hallmarks of mammalian ageing and mortality." Nature 654 (2026): 173-188.',
 'doi': 'https://doi.org/10.1038/s41586-026-10542-3',
 'notes': 'Elastic Net over 10,487 mouse-Entrez genes, multispecies multi-tissue, scaleddiff variant. Cohort-relative: predict_age runs the tAge cohort preprocessing on the raw RNA-seq counts itself, so a prediction is an age difference against the reference group rather than an absolute age -- zero means "looks like the reference", negative means younger-looking. Name the cohort\'s species with a 0/1 column among var_names (mouse, rat, macaque or human; absent or all-zero means mouse) and the samples to centre against with a truthy adata.obs["tage_reference_group"] (absent centres on the whole cohort); at least two samples are needed. The published pipeline\'